In [14]:
import numpy as np
import pandas as pd 
import cmath 
import scipy.special as sp
import scipy.signal as spg
import scipy.constants as const
import matplotlib.pyplot as plt
from scipy.constants import c, mu_0, epsilon_0, pi
from scipy.special import ai_zeros

I'm now looking at an alternative formulation to predict the location of TE and TM modes of the alumina disk, which essentially makes a modification to the following functions from Jackson's Classical Electrodynamics: 

TM modes: 

$$ 
\omega_{m,n,p} = \frac{c}{\sqrt{\mu\epsilon}} \sqrt{\frac{x_{m,n}^2}{R^2} + \frac{p^2 \pi^2}{d^2}}
$$

and TE modes: 

$$ 
\omega_{m,n,p} = \frac{c}{\sqrt{\mu\epsilon}} \sqrt{\frac{x\prime_{m,n}^2}{R^2} + \frac{p^2 \pi^2}{d^2}}
$$

which uses instead of Bessel functions, Airy functions, to account for the realities of a dielectric-air boundary and resonant modes with a system of containment/evanescent decay (turning point formulation), represented by: 

$$
f_{mnp} = \frac{1}{2\pi\sqrt{\mu\epsilon}} \sqrt{ \left( \frac{m+\alpha_n}{R} \right)^2 + \left( \frac{p\pi}{d} \right)^2 }
$$

The Airy root $\alpha_n$ is mode contingent; if it's a TM mode, it should use regular Ai($-\alpha_n^{TM}$), while in a TE mode it should use Ai'($\alpha_n^{TE}$). 


Building the function: 

In [15]:
pi

3.141592653589793

In [83]:
mu_0

1.25663706212e-06

In [84]:
epsilon_0

8.8541878128e-12

In [10]:
a, ap, ai, aip = ai_zeros(1)

In [11]:
print(a, ap, ai, aip)

[-2.33810741] [-1.01879297] [0.53565666] [0.70121082]


In [39]:
# get airy roots
alpha_1, alpha_prime_1, _, _, = ai_zeros(1)
alpha = (-1)*alpha_1
alpha_2 = (-1)*alpha_prime_1 

print(alpha, alpha_2)

[2.33810741] [1.01879297]


In [263]:
mu_0

1.25663706212e-06

In [265]:
4*pi*10**7

125663706.14359173

In [264]:
epsilon_0

8.8541878128e-12

In [299]:
def Airy_freq_solver(mode_kind, m, p, mu_r = 1, epsilon_r = 9.6, 
                     R = 0.076225, d = 0.02542957): 
                    
                    # get airy roots
                    alpha_1, alpha_prime_1, _, _, = ai_zeros(1)

                    # set material properties
                    mu = mu_0*mu_r
                    epsilon = epsilon_0*epsilon_r

                    # set TE or TM airy property
                    if mode_kind == 'TM': 
                        alpha = alpha_1 # regular Airy 
                    elif mode_kind == 'TE': 
                        alpha = alpha_prime_1 # derivative Airy

                    # calculate omega_mnp
                    omega_mnp = ((1/(np.sqrt(mu*epsilon))) * np.sqrt(((m + alpha)/R)**2 + (p**2*pi**2)/d**2))
                    freq = omega_mnp/(2*pi) # Hz
                    freq_GHz = freq/(10**9) # GHz

                    return freq_GHz

def get_TE_TM_m_p(N_m, p): 
    m_array = []
    TM_freqs = []
    TE_freqs = []
    for i in range(0, N_m): 
        TM_freq = Airy_freq_solver('TM', m = i, p=p)
        TE_freq = Airy_freq_solver('TE', m = i, p=p)
        m_array.append(i)
        TM_freqs.append(TM_freq)
        TE_freqs.append(TE_freq)
    eigenfreqs = {'m': m_array, 'TE': TE_freqs, 'TM': TM_freqs}
    eigen_m_table = pd.DataFrame(eigenfreqs)
    return eigen_m_table

In [300]:
first_test = get_TE_TM_m_p(N_m = 20, p=1)
first_test

,m,TE,TM
0,0,[1.913563723116242],[1.9602260392778548]
1,1,[1.902466153366491],[1.9215730219915155]
2,2,[1.9127618805707085],[1.903688219590846]
3,3,[1.944111047138098],[1.9071559980119668]
4,4,[1.995521666187839],[1.9318613741357784]
5,5,[2.065496279741623],[1.9770083312376892]
6,6,[2.152224952882152],[2.041240975355283]
7,7,[2.2537743495681584],[2.1228275252446296]
8,8,[2.3682387114720655],[2.2198553757359485]
9,9,[2.493840326692762],[2.3303965931171544]


In [262]:
first_test==second_test

,m,TE,TM
0,True,False,False
1,True,False,False
2,True,False,False
3,True,False,False
4,True,False,False
5,True,False,False
6,True,False,False
7,True,False,False
8,True,False,False
9,True,False,False


In [261]:
second_test = get_TE_TM_m_p(N_m = 20, p=1)
second_test

,m,TE,TM
0,0,[1.903674438707125],[1.9500956043336732]
1,1,[1.8926342211229592],[1.911642345579855]
2,2,[1.9028767400784565],[1.8938499717173123]
3,3,[1.9340638943645576],[1.8972998286829523]
4,4,[1.985208823681947],[1.9218775275896576]
5,5,[2.0548218089050456],[1.9667911655218993]
6,6,[2.1411022688479937],[2.030691855768109]
7,7,[2.2421268589371404],[2.11185676691814]
8,8,[2.355999669791087],[2.2083831781328245]
9,9,[2.4809521767118317],[2.318353119248491]


In [102]:
test = Airy_freq_solver('TE', m = 10, p = 1)

In [103]:
test

array([2.6073619])

In [224]:
# GPT VERSION

def wgm_frequency(mode_kind, m, p=0, epsilon_r=9.7598758464, mu_r=1.0, R=0.076225, d=0.02542957, n=1):
    """
    Calculate WGM frequencies using Airy approximation with TE/TM distinction.

    Parameters:
    ----------
    mode_kind : str
        'TE' or 'TM' for transverse electric or magnetic modes.
    m : int
        Azimuthal mode number (number of angular lobes).
    p : int, optional
        Axial mode number (default is 0 for both TE and TM).
    epsilon_r : float, optional
        Relative permittivity of dielectric (default alumina: 9.6).
    mu_r : float, optional
        Relative permeability of dielectric (default is 1.0).
    R : float, optional
        Radius of disk in meters (default: 0.0762 m = 3 inches).
    d : float, optional
        Thickness of disk in meters (default: 0.0254 m = 1 inch).
    n : int, optional
        Radial mode number (default is 1).

    Returns:
    -------
    freq_GHz : float
        Resonant frequency in GHz.
    """

    # Physical constants
    mu = mu_0 * mu_r
    epsilon = epsilon_0 * epsilon_r

    # Precomputed Airy roots (first few)
    airy_roots = [-2.3381, -4.0879, -5.5205]        # TM: Ai(-a_n)=0
    airy_deriv_roots = [-1.0188, -3.2482, -4.8201] # TE: Ai'(-a'_n)=0

    if n > len(airy_roots):
        raise ValueError(f"Radial mode n={n} is too high for precomputed Airy roots.")

    # Choose Airy correction
    if mode_kind.upper() == 'TM':
        alpha_n = airy_roots[n-1]
    elif mode_kind.upper() == 'TE':
        alpha_n = airy_deriv_roots[n-1]
    else:
        raise ValueError("mode_kind must be 'TE' or 'TM'.")

    # Correct radial quantization
    m_eff = m + alpha_n

    # Compute angular frequency
    term_radial = (m_eff / R)**2
    term_axial = (p * np.pi / d)**2

    omega = 1 / np.sqrt(mu * epsilon) * np.sqrt(term_radial + term_axial)

    # Convert to frequency in GHz
    freq_Hz = omega / (2 * np.pi)
    freq_GHz = freq_Hz * 1e-9

    return freq_GHz

In [227]:
wgm_frequency('TE', m=11, p=1, epsilon_r = 9.8)

2.743837579061404

In [209]:
def get_TE_TM_m_p(N_m, p): 
    m_array = []
    TM_freqs = []
    TE_freqs = []
    for i in range(0, N_m): 
        TM_freq = Airy_freq_solver('TM', m = i, p=p)
        TE_freq = Airy_freq_solver('TE', m = i, p=p)
        m_array.append(i)
        TM_freqs.append(TM_freq)
        TE_freqs.append(TE_freq)
    eigenfreqs = {'m': m_array, 'TE': TE_freqs, 'TM': TM_freqs}
    eigen_m_table = pd.DataFrame(eigenfreqs)
    return eigen_m_table




In [187]:
test2 = get_TE_TM_m_p(N_m = 20, p = 2)

In [188]:
test2

,m,TE,TM
0,0,[3.8104875373570803],[3.8341329234900137]
1,1,[3.826720937677024],[3.8642266659692863]
2,2,[3.85349194914671],[3.904554953098603]
3,3,[3.8905830506753087],[3.9548047034368525]
4,4,[3.9377026234907797],[4.014603381115793]
5,5,[3.9944957920457296],[4.083531508117352]
6,6,[4.060556676008427],[4.161135428304236]
7,7,[4.135441162713221],[4.246939572488745]
8,8,[4.2186794031106265],[4.340457650620932]
9,9,[4.309787396957939],[4.441202394180811]


In [210]:
def wgm_frequency_corrected(mode_kind, m, p=0, mu_r=1.0, epsilon_r=9.6, R=0.076225, d=0.02542957,
                         n=1, beta=0.7):
    """
    Improved WGM frequency model with effective index and polarization correction.
    """
    # Physical constants
    mu = mu_0 * mu_r
    epsilon = epsilon_0 * epsilon_r
    n_core = np.sqrt(epsilon_r)

    # Precomputed Airy roots (first few)
    airy_roots = [-2.3381, -4.0879, -5.5205]        # TM: Ai(-a_n)=0
    airy_deriv_roots = [-1.0188, -3.2482, -4.8201] # TE: Ai'(-a'_n)=0

    # Airy correction
    if mode_kind.upper() == 'TM':
        alpha_n = airy_roots[n-1]
        delta_polarization = -0.5  # Approximate polarization phase correction
    elif mode_kind.upper() == 'TE':
        alpha_n = airy_deriv_roots[n-1]
        delta_polarization = +0.5
    else:
        raise ValueError("mode_kind must be 'TE' or 'TM'.")

    # Effective refractive index correction
    n_eff = n_core - beta/m

    # Correct radial quantization
    m_eff = m - alpha_n + delta_polarization

    # Compute angular frequency
    term_radial = (m_eff / R)**2
    term_axial = (p * np.pi / d)**2

    omega = c / n_eff * np.sqrt(term_radial + term_axial)

    # Convert to frequency in GHz
    freq_Hz = omega / (2 * np.pi)
    freq_GHz = freq_Hz * 1e-9

    return freq_GHz

In [211]:
def get_TE_TM_m_p_GPT(N_m, p): 
    m_array = []
    TM_freqs = []
    TE_freqs = []
    for i in range(1, N_m): 
        TM_freq = wgm_frequency_corrected('TM', m = i, p=p)
        TE_freq = wgm_frequency_corrected('TE', m = i, p=p)
        m_array.append(i)
        TM_freqs.append(TM_freq)
        TE_freqs.append(TE_freq)
    eigenfreqs = {'m': m_array, 'TE': TE_freqs, 'TM': TM_freqs}
    eigen_m_table = pd.DataFrame(eigenfreqs)
    return eigen_m_table


In [212]:
test=get_TE_TM_m_p_GPT(20, p=1)

In [213]:
test

,m,TE,TM
0,1,2.544119,2.566915
1,2,2.289578,2.316035
2,3,2.282015,2.313051
3,4,2.337100,2.372402
4,5,2.423319,2.462398
5,6,2.529738,2.572095
6,7,2.650959,2.696131
7,8,2.783696,2.831272
8,9,2.925687,2.975313
9,10,3.075265,3.126640


In [206]:
# for TM 
def bessel_roots(m, n): 
    roots = sp.jn_zeros(m, n)
    return roots 

# for TE
def bessel_deriv_roots(m, n):
    roots = sp.jnp_zeros(m, n)
    return roots

# function to do whole thing

def bessel_omega_freq_solver(mode_kind, num_lobes, p_TE=1, mu = 1, 
                             epsilon = 9.7598758464, d = 0.02542957, 
                             R=0.076225, N = 1, n = 0): # epsilon = 9.7598758464 D = 152.44 mm # d = 0.02542957, R = 0.07622
    # define constants
    m = int(num_lobes // 2)
    
    if mode_kind == 'TM': # solve bessel and define p for transverse magnetic (TM)
        x_mn = bessel_roots(m,N)[n]
        p = 0
    if mode_kind == 'TE': # solve bessel and define p for transverse electric (TE) 
        x_mn = bessel_deriv_roots(m,N)[n]
        p = p_TE

    # calculate omega_mnp
    omega_mnp = (c/np.sqrt(mu*epsilon)) * (np.sqrt(((x_mn**2)/(R**2)) + (p**2*np.pi**2)/d**2))
    freq = omega_mnp/(2*np.pi) # Hz
    freq_GHz = freq/(10**9) # GHz

    return freq_GHz

def get_TE_TM_m_p(N_m, p_TE=1): 
    m_array = []
    TM_freqs = []
    TE_freqs = []
    for i in range(0, N_m): 
        TM_freq = bessel_omega_freq_solver('TM', num_lobes = i*2)
        TE_freq = bessel_omega_freq_solver('TE', num_lobes = i*2, p_TE=p_TE)
        m_array.append(i)
        TM_freqs.append(TM_freq)
        TE_freqs.append(TE_freq)
    eigenfreqs = {'m': m_array, 'TE': TE_freqs, 'TM': TM_freqs}
    eigen_m_table = pd.DataFrame(eigenfreqs)
    return eigen_m_table


# ------ constants --------

c = const.c

test_TE = get_TE_TM_m_p(40, p_TE=1)
test_TE

,m,TE,TM
0,0,2.037031,0.481842
1,1,1.922542,0.767739
2,2,1.983575,1.028998
3,3,2.066071,1.278359
4,4,2.166854,1.520436
5,5,2.283088,1.757496
6,6,2.412271,1.990846
7,7,2.552252,2.221317
8,8,2.701217,2.449477
9,9,2.857660,2.675731


In [284]:
import numpy as np
from scipy.special import jv, jvp, hankel1, h1vp
from scipy.constants import c
from scipy.optimize import root_scalar

def dispersion_relation_TE(omega, m, R, epsilon1, epsilon2, mu1, mu2, p, d):
    """
    Dispersion relation for TE modes.
    """
    k0 = omega / c
    k1 = np.lib.scimath.sqrt(epsilon1 * mu1 * k0**2 - (p*np.pi/d)**2)
    k2 = np.lib.scimath.sqrt(epsilon2 * mu2 * k0**2 - (p*np.pi/d)**2)

    lhs = jvp(m, k1*R) / jv(m, k1*R)
    rhs = (epsilon1 * k2) / (epsilon2 * k1) * h1vp(m, k2*R) / hankel1(m, k2*R)

    return np.real(lhs - rhs)  # Root finder needs real values

def dispersion_relation_TM(omega, m, R, epsilon1, epsilon2, mu1, mu2, p, d):
    """
    Dispersion relation for TM modes.
    """
    k0 = omega / c
    k1 = np.lib.scimath.sqrt(epsilon1 * mu1 * k0**2 - (p*np.pi/d)**2)
    k2 = np.lib.scimath.sqrt(epsilon2 * mu2 * k0**2 - (p*np.pi/d)**2)

    lhs = jvp(m, k1*R) / jv(m, k1*R)
    rhs = (mu1 * k2) / (mu2 * k1) * h1vp(m, k2*R) / hankel1(m, k2*R)

    return np.real(lhs - rhs)  # Root finder needs real values

def find_mode_frequency(mode_kind, m, R, epsilon1, epsilon2=1.0, mu1=1.0, mu2=1.0, p=0, d=0.0254, f_scan=(1.0,10.0)):
    """
    Find mode frequency in GHz.
    """
    # Set up the dispersion function
    if mode_kind.upper() == "TE":
        disp_func = lambda omega: dispersion_relation_TE(omega, m, R, epsilon1, epsilon2, mu1, mu2, p, d)
    elif mode_kind.upper() == "TM":
        disp_func = lambda omega: dispersion_relation_TM(omega, m, R, epsilon1, epsilon2, mu1, mu2, p, d)
    else:
        raise ValueError("mode_kind must be 'TE' or 'TM'.")

    # Scan for sign changes to find a bracket
    f_min, f_max = f_scan
    omegas = np.linspace(2*np.pi*f_min*1e9, 2*np.pi*f_max*1e9, 10000)
    values = [disp_func(omega) for omega in omegas]
    
    for i in range(len(values)-1):
        if np.sign(values[i]) != np.sign(values[i+1]):
            # Found sign change: use as bracket
            bracket = [omegas[i], omegas[i+1]]
            sol = root_scalar(disp_func, bracket=bracket, method="brentq")
            if sol.converged:
                freq_GHz = sol.root / (2*np.pi*1e9)
                return freq_GHz

    raise RuntimeError("No root found in scan range.")

# Parameters for alumina disk
R = 0.076225  # radius in meters (3 inches)
epsilon1 = 9.7598758464
d = 0.02542957
m = 8

# Find TE mode
freq_TE = find_mode_frequency('TE', m=m, R=R, epsilon1=epsilon1, d=d, p=1, f_scan=(1,10))
print(f"TE mode (m={m}): {freq_TE:.6f} GHz")

# Find TM mode
freq_TM = find_mode_frequency('TM', m=m, R=R, epsilon1=epsilon1, d=d, p=1, f_scan=(1,10))
print(f"TM mode (m={m}): {freq_TM:.6f} GHz")


TE mode (m=8): 3.074922 GHz
TM mode (m=8): 2.954904 GHz


In [293]:
modes = []
for i in range(20): 
    freq_TE = find_mode_frequency('TE', m=i, R=R, epsilon1=9.9, d=d, p=1, f_scan=(1,10))
    freq_TM = find_mode_frequency('TM', m=i, R=R, epsilon1=9.9, d=d, p=1, f_scan=(1,10))
    modes.append([i, freq_TE, freq_TM])

TEs_df = pd.DataFrame(modes, columns=['m', 'TE', 'TM'])
    

In [294]:
TEs_df

,m,TE,TM
0,0,1.932278,1.922336
1,1,1.873415,1.873415
2,2,2.128738,2.089587
3,3,1.873415,1.873415
4,4,2.396395,2.325919
5,5,1.873415,1.873415
6,6,2.709910,2.612182
7,7,1.873415,1.873415
8,8,3.053342,2.934041
9,9,1.873415,1.873415
